In [1]:
import pandas as pd

capital_bikeshare_data_2026   = pd.read_csv('Capital_Bike_Share_Locations.csv')
washington_dc_crime_data_2026 = pd.read_csv('Crime_Incidents_in_2026.csv')

df_bike = capital_bikeshare_data_2026[['LATITUDE','LONGITUDE']].copy()
df_crime = washington_dc_crime_data_2026[['LATITUDE','LONGITUDE', 'OFFENSE']].copy()

print(df_bike.head())
print(df_crime.head())

    LATITUDE  LONGITUDE
0  38.880612 -77.171891
1  39.083673 -77.149162
2  38.964544 -77.075135
3  38.891805 -76.913563
4  38.883318 -76.925315
    LATITUDE  LONGITUDE              OFFENSE
0  38.951866 -76.996990  MOTOR VEHICLE THEFT
1  38.952097 -77.072194              ROBBERY
2  38.825089 -77.001446          THEFT/OTHER
3  38.847782 -76.971907          THEFT/OTHER
4  38.907206 -76.930151          THEFT/OTHER


In [ ]:
from math import pi, radians, sin, cos, asin, sqrt, floor
EARTH_RADIUS_KM = 6371
KM_PER_DEG = 2 * pi * EARTH_RADIUS_KM / 360
DC_BLOCK_RADIUS_KM = 62 / 1000 # 12,000m^2 approx block size, then sqrt(x/pi) to get radius
BLOCKS = 5
CIRCLE_RADIUS_KM = DC_BLOCK_RADIUS_KM * BLOCKS # ~BLOCKSm block radius
CELL_DEG = CIRCLE_RADIUS_KM / KM_PER_DEG

def distance_earth(lat1, lon1, lat2, lon2):
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = (sin(dlat / 2) ** 2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2) ** 2)
    return 2 * EARTH_RADIUS_KM * asin(sqrt(a))

def get_grid_cell(lat, lon):
    i = floor(lat / CELL_DEG)
    j = floor(lon / CELL_DEG)
    return (i, j)

def point_in_any_circle(lat, lon, grid):
    i, j = get_grid_cell(lat, lon)
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            for clat, clon in grid.get((i + di, j + dj), []):
                if distance_earth(lat, lon, clat, clon) <= CIRCLE_RADIUS_KM:
                    return True
    return False

In [3]:
# fill grid
grid = {}
for i, row in df_bike.iterrows():
    lat, long = row['LATITUDE'], row['LONGITUDE']
    x, y = get_grid_cell(lat, long)
    grid.setdefault((x, y), []).append((lat, long))

# print head of grid
print(len(grid))
for k, v in list(grid.items())[:5]:
    print(k, v)

bucket_sizes = [len(v) for v in grid.values()]
print("nonempty buckets:", len(bucket_sizes))
print("avg per bucket:", sum(bucket_sizes) / len(bucket_sizes))
print("max bucket size:", max(bucket_sizes))
print("min bucket size:", min(bucket_sizes))
    

763
(13946, -27682) [(38.880612, -77.171891)]
(14019, -27673) [(39.083673, -77.149162), (39.084379, -77.146866)]
(13976, -27647) [(38.964544, -77.075135)]
(13950, -27589) [(38.891805, -76.913563)]
(13947, -27593) [(38.883318, -76.925315)]
nonempty buckets: 763
avg per bucket: 1.073394495412844
max bucket size: 3
min bucket size: 1


In [4]:
# fill in circles to crime data
df_crime['bike_nearby'] = False

df_crime['bike_nearby'] = df_crime.apply(
    lambda row: point_in_any_circle(row['LATITUDE'], row['LONGITUDE'], grid),
    axis=1
)

print(df_crime.head())
df_crime['bike_nearby'].value_counts()

    LATITUDE  LONGITUDE              OFFENSE  bike_nearby
0  38.951866 -76.996990  MOTOR VEHICLE THEFT        False
1  38.952097 -77.072194              ROBBERY        False
2  38.825089 -77.001446          THEFT/OTHER         True
3  38.847782 -76.971907          THEFT/OTHER        False
4  38.907206 -76.930151          THEFT/OTHER         True


bike_nearby
True     3708
False    1093
Name: count, dtype: int64

In [5]:
print(df_crime['OFFENSE'].unique())

VIOLENT  = ['ROBBERY', 'HOMICIDE', 'ASSAULT W/DANGEROUS WEAPON', 'SEX ABUSE']
PROPERTY = ['MOTOR VEHICLE THEFT', 'THEFT/OTHER', 'THEFT F/AUTO', 'BURGLARY']

df_crime['is_violent']  = df_crime['OFFENSE'].isin(VIOLENT)
df_crime['is_property'] = df_crime['OFFENSE'].isin(PROPERTY)

print(f"Violent crimes:  {df_crime['is_violent'].sum()}")
print(f"Property crimes: {df_crime['is_property'].sum()}")
print(f"Total crimes:    {len(df_crime)}")

['MOTOR VEHICLE THEFT' 'ROBBERY' 'THEFT/OTHER' 'THEFT F/AUTO' 'BURGLARY'
 'ASSAULT W/DANGEROUS WEAPON' 'SEX ABUSE' 'HOMICIDE']
Violent crimes:  598
Property crimes: 4203
Total crimes:    4801


In [6]:
# crime breakdown by proximity (bike_nearby) 

near = df_crime[df_crime['bike_nearby'] == True]
far  = df_crime[df_crime['bike_nearby'] == False]

prop_near = near['is_property'].mean() * 100
prop_far  = far['is_property'].mean() * 100
viol_near = near['is_violent'].mean() * 100
viol_far  = far['is_violent'].mean() * 100

print(f"Crimes near a bikeshare station: {len(near)}")
print(f"Crimes far from any station:     {len(far)}")
print()
print(f"Property crime rate : Near: {prop_near:.1f}%   Far: {prop_far:.1f}%")
print(f"Violent  crime rate : Near: {viol_near:.1f}%   Far: {viol_far:.1f}%")

Crimes near a bikeshare station: 3708
Crimes far from any station:     1093

Property crime rate : Near: 88.9%   Far: 83.1%
Violent  crime rate : Near: 11.1%   Far: 16.9%


In [7]:
from scipy.stats import chi2_contingency
import numpy as np

alpha = 0.05

# ===============================================================
#  SET 1: Property Crime
#  Ho: There is no association between bike_nearby and property crime rate (p = 0)
#  Ha: Closer proximity correlates with higher property crime rate
# ===============================================================

ct_prop = pd.crosstab(df_crime['bike_nearby'], df_crime['is_property'])
chi2_prop, p_prop, dof_prop, expected_prop = chi2_contingency(ct_prop)

print("=" * 65)
print("SET 1: Property Crime: Chi-Squared Test of Independence")
print("=" * 65)
print(f"\nContingency table:\n{ct_prop}\n")
print(f"  Chi2 = {chi2_prop:.4f},  df = {dof_prop},  p-value = {p_prop:.4e}")
print(f"  Near bikeshare: {prop_near:.1f}% property crime")
print(f"  Far from bikeshare: {prop_far:.1f}% property crime")
if p_prop < alpha:
    print(f"\n  -> Reject Ho at a = {alpha}.")
    if prop_near > prop_far:
        print("    Property crime rate is significantly HIGHER near bikeshare stations (supports Ha).")
    else:
        print("    Property crime rate is significantly LOWER near bikeshare stations (opposite of Ha).")
else:
    print(f"\n  -> Fail to reject Ho at a = {alpha}. No significant association found.")

SET 1: Property Crime: Chi-Squared Test of Independence

Contingency table:
is_property  False  True 
bike_nearby              
False          185    908
True           413   3295

  Chi2 = 25.4054,  df = 1,  p-value = 4.6463e-07
  Near bikeshare: 88.9% property crime
  Far from bikeshare: 83.1% property crime

  -> Reject Ho at a = 0.05.
    Property crime rate is significantly HIGHER near bikeshare stations (supports Ha).


In [8]:
# ===============================================================
#  SET 2: Violent Crime
#  Ho: There is no association between bike_nearby and violent crime rate (p = 0)
#  Ha: Closer proximity correlates with lower violent crime rate
# ===============================================================

ct_viol = pd.crosstab(df_crime['bike_nearby'], df_crime['is_violent'])
chi2_viol, p_viol, dof_viol, expected_viol = chi2_contingency(ct_viol)

print("=" * 65)
print("SET 2: Violent Crime: Chi-Squared Test of Independence")
print("=" * 65)
print(f"\nContingency table:\n{ct_viol}\n")
print(f"  Chi2 = {chi2_viol:.4f},  df = {dof_viol},  p-value = {p_viol:.4e}")
print(f"  Near bikeshare: {viol_near:.1f}% violent crime")
print(f"  Far from bikeshare: {viol_far:.1f}% violent crime")
if p_viol < alpha:
    print(f"\n  -> Reject Ho at a = {alpha}.")
    if viol_near < viol_far:
        print("    Violent crime rate is significantly LOWER near bikeshare stations (supports Ha).")
    else:
        print("    Violent crime rate is significantly HIGHER near bikeshare stations (opposite of Ha).")
else:
    print(f"\n  -> Fail to reject Ho at a = {alpha}. No significant association found.")

SET 2: Violent Crime: Chi-Squared Test of Independence

Contingency table:
is_violent   False  True 
bike_nearby              
False          908    185
True          3295    413

  Chi2 = 25.4054,  df = 1,  p-value = 4.6463e-07
  Near bikeshare: 11.1% violent crime
  Far from bikeshare: 16.9% violent crime

  -> Reject Ho at a = 0.05.
    Violent crime rate is significantly LOWER near bikeshare stations (supports Ha).


In [9]:
# =======================================================================
#  CONCLUSION
# =======================================================================
#
#  Research question:
#    Does proximity to a Capital Bikeshare station correlate with
#    higher rates of property crime but lower rates of violent crime?
#
#  Methodology:
#    - Each crime was classified as "near" (within ~5 DC-blocks of a
#      bikeshare station) or "far" using the bike_nearby flag.
#    - Chi-squared tests of independence tested whether crime-type
#      proportions differ between the two groups at a = 0.05.
#
#  SET 1: Property Crime:
#    Near: {prop_near}%  vs  Far: {prop_far}%
#    Chi2 test p-value = {p_prop}
#    -> Result printed above.
#
#  SET 2: Violent Crime:
#    Near: {viol_near}%  vs  Far: {viol_far}%
#    Chi2 test p-value = {p_viol}
#    -> Result printed above.
#

print("=" * 65)
print("CONCLUSION")
print("=" * 65)
print()

if p_prop < alpha and prop_near > prop_far:
    print("SET 1: Property crime rate IS significantly higher near")
    print("       bikeshare stations. (YES) Supports the hypothesis.")
elif p_prop < alpha:
    print("SET 1: Property crime rate IS significantly different, but")
    print("       LOWER near stations. (NO) Opposite of hypothesis.")
else:
    print("SET 1: No significant difference in property crime rates.")

print()

if p_viol < alpha and viol_near < viol_far:
    print("SET 2: Violent crime rate IS significantly lower near")
    print("       bikeshare stations. (YES) Supports the hypothesis.")
elif p_viol < alpha:
    print("SET 2: Violent crime rate IS significantly different, but")
    print("       HIGHER near stations. (NO) Opposite of hypothesis.")
else:
    print("SET 2: No significant difference in violent crime rates.")

CONCLUSION

SET 1: Property crime rate IS significantly higher near
       bikeshare stations. (YES) Supports the hypothesis.

SET 2: Violent crime rate IS significantly lower near
       bikeshare stations. (YES) Supports the hypothesis.
